# Part 4 - GenAI Notes

## Part 4 - GenAI notes
CRAFT: Context=call-centre note for missed appointment, Role=clinic analyst, Action=classify single reason, Format=JSON {root_cause,confidence,summary,recommended_action}, Tone=plain minimal. Allowed ONLY: forgot_or_no_reminder, distance_or_transport, cost_or_fee, illness_or_emergency, work_or_schedule, other. Temp 0, Gemini Flash, response_schema + Pydantic. N004 blank -> skipped_blank, no API call (9 calls for 10 notes).
### 4.3 Break it
(a) confidence 1.7 fails 'less than or equal to 1', missing recommended_action fails 'Field required / missing'. (b) N007 Hinglish + N009 ambiguous at temp 0 vs temp 1, and plain prompt (no CRAFT, plain string, temp 0) - V2 still returns allowed root_cause for both, less consistent on N009. (c) Calls: 1 + 9 + 4 = 14 total, <25.
### 4.4 Merged
Merge 10 notes with appointments_clean: 5/10 reminder_sent=No. Both forgot notes N001,N007 are No (2/2).

In [ ]:
from google import genai
from google.genai import types
import json, pandas as pd, time
from pydantic import BaseModel, Field
SYSTEM='You are a clinic analyst. Classify the single most likely reason this patient missed their appointment. Allowed root_cause ONLY: forgot_or_no_reminder, distance_or_transport, cost_or_fee, illness_or_emergency, work_or_schedule, other. Return confidence 0-1, summary at most 25 words, recommended_action. If ambiguous, choose other rather than guess.'
SCHEMA={'type':'object','properties':{'root_cause':{'type':'string'},'confidence':{'type':'number'},'summary':{'type':'string'},'recommended_action':{'type':'string'}},'required':['root_cause','confidence','summary','recommended_action']}
class FollowupInsight(BaseModel):
    root_cause: str = Field(min_length=3); confidence: float = Field(ge=0, le=1); summary: str = Field(min_length=5); recommended_action: str = Field(min_length=5)
client=genai.Client(api_key=input('Paste your key: '))
def analyse(note, temp=0):
    r=client.models.generate_content(model='gemini-2.5-flash', contents='Call-centre note:\n'+note+'\n\nClassify it using allowed values only.', config=types.GenerateContentConfig(system_instruction=SYSTEM, temperature=temp, max_output_tokens=1000, response_mime_type='application/json', response_schema=SCHEMA, thinking_config=types.ThinkingConfig(thinking_level='minimal')))
    return json.loads(r.text)
notes=pd.read_csv('data/call_centre_notes.csv')
print(analyse(notes[notes.note_id=='N001'].iloc[0].note_text))
rows=[]
for _,r in notes.head(10).iterrows():
    if pd.isna(r.note_text) or str(r.note_text).strip()=='':
        rows.append({'note_id':r.note_id,'appointment_id':r.appointment_id,'root_cause':'other','confidence':0.0,'summary':'No note recorded.','recommended_action':'Re-contact patient.','validation_status':'skipped_blank','raw_response':''}); continue
    raw=analyse(str(r.note_text)); raw_s=json.dumps(raw)
    try:
        assert raw.get('root_cause') in ['forgot_or_no_reminder','distance_or_transport','cost_or_fee','illness_or_emergency','work_or_schedule','other']; obj=FollowupInsight(**raw)
        rows.append({'note_id':r.note_id,'appointment_id':r.appointment_id,'root_cause':obj.root_cause,'confidence':obj.confidence,'summary':obj.summary,'recommended_action':obj.recommended_action,'validation_status':'valid','raw_response':raw_s})
    except Exception:
        rows.append({'note_id':r.note_id,'appointment_id':r.appointment_id,'root_cause':raw.get('root_cause',''),'confidence':raw.get('confidence',''),'summary':raw.get('summary',''),'recommended_action':raw.get('recommended_action',''),'validation_status':'invalid','raw_response':raw_s})
    time.sleep(13)
pd.DataFrame(rows).to_csv('notes_analysed.csv',index=False); print(pd.DataFrame(rows))